# Hugging Face: final participant notebook
## 24 lightweight tasks / 48 examples

### Start here in Google Colab
1. Run **STEP 1**. It installs the tested package versions. If versions change, the session restarts once automatically; the disconnection is intentional.
2. After Colab reconnects, run **STEP 2**, then **STEP 3** and upload `input_files.zip`.
3. Run **STEP 4** for Tesseract, then select the examples you want.
4. Keep the pinned versions in STEP 1. Do not run a separate `pip install -U transformers` in this notebook.

Each example directly loads its pipeline or model. No generic model runner is used. Most examples run on CPU. Later examples using the same model reuse its cached download in the same runtime. The models are compact task-specific checkpoints; none are multi-billion-parameter models. First downloads still depend on internet speed.

### Tested environment and result
**Python 3.13.15 · PyTorch 2.8.0 CPU · Transformers 4.57.1**

All **48 example cells** and **5 setup/check cells** completed without runtime exceptions in a clean Linux environment using the supplied input ZIP. The revised sentiment inputs were also rechecked. The Colab upload dialog itself requires Colab and was not exercised in the local test.

The active Transformers version and required pipeline task names are checked before examples run. This prevents the `Unknown task question-answering` error from an incompatible Transformers version. Package installation happens only at the beginning.

The banking documents/tables are fictional. Model predictions can still be inaccurate; compare them with the source data. Ordinary framework warnings are different from runtime exceptions. Public model access does not require an API key; the optional Hub authentication warning is not a failure.


In [ ]:
# STEP 1: install the tested package versions before importing Transformers.
# A one-time session restart after installation is intentional.
import sys
import os
import signal
import subprocess
from importlib.metadata import version, PackageNotFoundError

requirements = ['transformers==4.57.1', 'huggingface_hub==0.36.0', 'accelerate==1.11.0', 'sentence-transformers==5.1.1', 'chronos-forecasting==1.5.3', 'silero-vad==6.0.0', 'onnxruntime==1.23.2', 'numpy==2.2.6', 'pandas==2.3.3', 'Pillow==11.3.0', 'soundfile==0.13.1', 'librosa==0.11.0', 'scipy==1.16.3', 'sentencepiece==0.2.1', 'safetensors==0.6.2', 'pytesseract==0.3.13']
pytorch_packages = ["torch==2.8.0", "torchvision==0.23.0", "torchaudio==2.8.0"]

def needs_install(packages):
    for item in packages:
        name, wanted = item.split("==")
        try:
            if version(name).split("+")[0] != wanted:
                return True
        except PackageNotFoundError:
            return True
    return False

changed = False
if needs_install(pytorch_packages):
    # CPU wheels keep the package download smaller than CUDA wheels.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pytorch_packages +
                          ["--index-url", "https://download.pytorch.org/whl/cpu"])
    changed = True

if needs_install(requirements):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + requirements)
    changed = True

loaded = sys.modules.get("transformers")
wrong_loaded_version = loaded is not None and loaded.__version__ != "4.57.1"
if changed or wrong_loaded_version:
    print("Setup complete. Restarting the session once. After reconnection, run from STEP 2.", flush=True)
    os.kill(os.getpid(), signal.SIGKILL)
else:
    print("Tested package versions are installed. Continue with STEP 2.")

In [ ]:
# STEP 2: check the active version, then import the libraries.
import os
os.environ["USE_TF"] = "0"  # This notebook uses PyTorch only.
os.environ["HF_HUB_DISABLE_XET"] = "1"
import transformers
assert transformers.__version__ == "4.57.1", "Run STEP 1 and let the session restart before continuing."
from transformers.pipelines import get_supported_tasks
assert "question-answering" in get_supported_tasks()
assert "text2text-generation" in get_supported_tasks()

import os
import gc
import zipfile
import torch

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from PIL import Image, ImageDraw
from IPython.display import display, Audio, Video
from transformers import pipeline

os.makedirs("outputs", exist_ok=True)
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("Transformers:", transformers.__version__)
print("Environment check passed.")
torch.set_num_threads(2)

In [ ]:
# STEP 3: upload and extract input_files.zip.
import os
import zipfile

if not os.path.exists("input_files.zip"):
    from google.colab import files
    files.upload()

with zipfile.ZipFile("input_files.zip") as archive:
    has_input_folder = any(name.startswith("input_files/") for name in archive.namelist())
    archive.extractall("." if has_input_folder else "input_files")

os.makedirs("outputs", exist_ok=True)
required = ["bank_policy.png", "loan_policy.png", "bank_policy.txt", "loan_policy.txt",
            "cats.png", "parrots.png", "branch_table.csv", "daily_series.csv",
            "speech_1.wav", "speech_2.wav", "silence.wav"]
missing = [name for name in required if not os.path.exists("input_files/" + name)]
assert not missing, "These files are missing from input_files.zip: " + str(missing)
print("All required input files are ready.")

In [ ]:
# STEP 4: check/install Tesseract for the two document OCR examples.
import shutil
import subprocess

if shutil.which("tesseract") is None:
    subprocess.check_call(["apt-get", "update", "-qq"])
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "tesseract-ocr"])
print("Tesseract is ready. You can now run the examples.")

### Input files

Use the previously supplied `input_files.zip`. It includes all files used below; no extra sample downloads are needed. Banking documents and tables are synthetic training data. Public photo and speech attribution is included in the ZIP.


## Task index

| No. | Task |
|---|---|
| 1 | Document Question Answering |
| 2 | Depth Estimation |
| 3 | Image Classification |
| 4 | Object Detection |
| 5 | Image Segmentation |
| 6 | Image-to-Image |
| 7 | Image Feature Extraction |
| 8 | Text Classification |
| 9 | Token Classification |
| 10 | Table Question Answering |
| 11 | Question Answering |
| 12 | Zero-Shot Classification |
| 13 | Translation |
| 14 | Summarization |
| 15 | Feature Extraction |
| 16 | Text Generation |
| 17 | Fill-Mask |
| 18 | Sentence Similarity |
| 19 | Text Ranking |
| 20 | Text-to-Speech |
| 21 | Automatic Speech Recognition |
| 22 | Audio-to-Audio |
| 23 | Voice Activity Detection |
| 24 | Time Series Forecasting |

## 01. Document Question Answering

**Lightweight example.**

Read text from a document PNG with Tesseract OCR, then use a compact question-answering pipeline to extract an answer. This simplified workflow does not model the page layout. The supplied PDFs already have matching PNG pages.

**API:** OCR + `question-answering` pipeline. **Runtime:** CPU.


### Example 1: What is the minimum balance?

Input: a rendered document page. Output: extracted answer, OCR word positions and confidence.

In [ ]:
import pytesseract
image = Image.open("input_files/bank_policy.png")
context = pytesseract.image_to_string(image)
pipe = pipeline("question-answering", model="deepset/minilm-uncased-squad2")
print(pipe(question='What is the minimum balance?', context=context))

# Release this example model before opening the next example.
del pipe
gc.collect()


### Example 2: What is the processing fee?

Input: a rendered document page. Output: extracted answer, OCR word positions and confidence.

In [ ]:
import pytesseract
image = Image.open("input_files/loan_policy.png")
context = pytesseract.image_to_string(image)
pipe = pipeline("question-answering", model="deepset/minilm-uncased-squad2")
print(pipe(question='What is the processing fee?', context=context))

# Release this example model before opening the next example.
del pipe
gc.collect()


## 02. Depth Estimation

**Lightweight example.**

**Category:** Computer Vision  
**What it does:** Estimates relative distance from a single image.  
**API:** Transformers pipeline  
**Runtime:** CPU supported; no GPU required.



[Official reference](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines)

Smaller checkpoints prioritise download size and speed over accuracy. [Model references](#smaller-model-references).

### Example 1: cats.png

Bright/dark depth values encode relative depth. They are not calibrated metres.

In [ ]:
pipe = pipeline('depth-estimation', model='depth-anything/Depth-Anything-V2-Small-hf')
result = pipe("input_files/cats.png")
display(result["depth"])

# Release this example model before opening the next example.
del pipe
gc.collect()

### Example 2: parrots.png

Bright/dark depth values encode relative depth. They are not calibrated metres.

In [ ]:
pipe = pipeline('depth-estimation', model='depth-anything/Depth-Anything-V2-Small-hf')
result = pipe("input_files/parrots.png")
display(result["depth"])

# Release this example model before opening the next example.
del pipe
gc.collect()

## 03. Image Classification

**Lightweight example.**

**Category:** Computer Vision  
**What it does:** Assigns a whole-image label from fixed learned classes.  
**API:** Transformers pipeline  
**Runtime:** CPU supported; no GPU required.



[Official reference](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines)

Smaller checkpoints prioritise download size and speed over accuracy. [Model references](#smaller-model-references).

### Example 1: cats.png

Output: top labels and scores. An image can contain objects outside the model’s classes.

In [ ]:
pipe = pipeline('image-classification', model='WinKawaks/vit-tiny-patch16-224')
result = pipe("input_files/cats.png", top_k=3)
print(result)

# Release this example model before opening the next example.
del pipe
gc.collect()

### Example 2: parrots.png

Output: top labels and scores. An image can contain objects outside the model’s classes.

In [ ]:
pipe = pipeline('image-classification', model='WinKawaks/vit-tiny-patch16-224')
result = pipe("input_files/parrots.png", top_k=3)
print(result)

# Release this example model before opening the next example.
del pipe
gc.collect()

## 04. Object Detection

**Lightweight example.**

**Category:** Computer Vision  
**What it does:** Locates known object classes using bounding boxes.  
**API:** Transformers pipeline  
**Runtime:** CPU supported; no GPU required.



[Official reference](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines)

Smaller checkpoints prioritise download size and speed over accuracy. [Model references](#smaller-model-references).

### Example 1: cats.png

Output: object labels, confidence scores, and bounding boxes drawn on the image.

In [ ]:
pipe = pipeline('object-detection', model='hustvl/yolos-tiny')
result = pipe(Image.open("input_files/cats.png").convert("RGB").resize((320, 320)), threshold=0.7)
print(result)
image = Image.open("input_files/cats.png").convert("RGB").resize((320, 320))
draw = ImageDraw.Draw(image)
for item in result:
    box = item["box"]
    draw.rectangle([box["xmin"], box["ymin"], box["xmax"], box["ymax"]], outline="red", width=3)
    draw.text((box["xmin"], box["ymin"]), item["label"], fill="red")
display(image)

# Release this example model before opening the next example.
del pipe
gc.collect()

### Example 2: parrots.png

Output: object labels, confidence scores, and bounding boxes drawn on the image.

In [ ]:
pipe = pipeline('object-detection', model='hustvl/yolos-tiny')
result = pipe(Image.open("input_files/parrots.png").convert("RGB").resize((320, 320)), threshold=0.7)
print(result)
image = Image.open("input_files/parrots.png").convert("RGB").resize((320, 320))
draw = ImageDraw.Draw(image)
for item in result:
    box = item["box"]
    draw.rectangle([box["xmin"], box["ymin"], box["xmax"], box["ymax"]], outline="red", width=3)
    draw.text((box["xmin"], box["ymin"]), item["label"], fill="red")
display(image)

# Release this example model before opening the next example.
del pipe
gc.collect()

## 05. Image Segmentation

**Lightweight example.**

**Category:** Computer Vision  
**What it does:** Assigns semantic labels to pixels.  
**API:** Transformers pipeline  
**Runtime:** CPU supported; no GPU required.



[Official reference](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines)

Smaller checkpoints prioritise download size and speed over accuracy. [Model references](#smaller-model-references).

### Example 1: cats.png

Output: labelled pixel masks. The first mask is displayed; a model may return several masks.

In [ ]:
pipe = pipeline('image-segmentation', model='nvidia/segformer-b0-finetuned-ade-512-512')
result = pipe(Image.open("input_files/cats.png").convert("RGB").resize((320, 320)))
print([item["label"] for item in result])
if result:
    display(result[0]["mask"])

# Release this example model before opening the next example.
del pipe
gc.collect()

### Example 2: parrots.png

Output: labelled pixel masks. The first mask is displayed; a model may return several masks.

In [ ]:
pipe = pipeline('image-segmentation', model='nvidia/segformer-b0-finetuned-ade-512-512')
result = pipe(Image.open("input_files/parrots.png").convert("RGB").resize((320, 320)))
print([item["label"] for item in result])
if result:
    display(result[0]["mask"])

# Release this example model before opening the next example.
del pipe
gc.collect()

## 06. Image-to-Image

**Lightweight example.**

**Category:** Computer Vision  
**What it does:** Transforms a source image; this example uses learned super-resolution.  
**API:** Transformers pipeline  
**Runtime:** CPU supported; no GPU required.



[Official reference](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines)

Smaller checkpoints prioritise download size and speed over accuracy. [Model references](#smaller-model-references).

### Example 1: cats.png

Input: a 128×128 image. Output: a learned 2× enlarged image. This is a genuine image-to-image task without a text prompt.

In [ ]:
image = Image.open("input_files/cats.png").convert("RGB").resize((128, 128))
pipe = pipeline('image-to-image', model='caidas/swin2SR-classical-sr-x2-64')
result = pipe(image)
print("Input size:", image.size, "Output size:", result.size)
display(result)

# Release this example model before opening the next example.
del pipe
gc.collect()

### Example 2: parrots.png

Input: a 128×128 image. Output: a learned 2× enlarged image. This is a genuine image-to-image task without a text prompt.

In [ ]:
image = Image.open("input_files/parrots.png").convert("RGB").resize((128, 128))
pipe = pipeline('image-to-image', model='caidas/swin2SR-classical-sr-x2-64')
result = pipe(image)
print("Input size:", image.size, "Output size:", result.size)
display(result)

# Release this example model before opening the next example.
del pipe
gc.collect()

## 07. Image Feature Extraction

**Lightweight example.**

**Category:** Computer Vision  
**What it does:** Converts image patches into numerical features.  
**API:** Transformers pipeline  
**Runtime:** CPU supported; no GPU required.



[Official reference](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines)

Smaller checkpoints prioritise download size and speed over accuracy. [Model references](#smaller-model-references).

### Example 1: cats.png

Output: feature arrays. These can support similarity or clustering; they do not directly contain class names.

In [ ]:
pipe = pipeline('image-feature-extraction', model='WinKawaks/vit-tiny-patch16-224', model_kwargs={'add_pooling_layer': False})
result = pipe(Image.open("input_files/cats.png").convert("RGB").resize((320, 320)))
features = np.asarray(result)
print("Feature shape:", features.shape)
print("First values:", features.reshape(-1)[:8])

# Release this example model before opening the next example.
del pipe
gc.collect()

### Example 2: parrots.png

Output: feature arrays. These can support similarity or clustering; they do not directly contain class names.

In [ ]:
pipe = pipeline('image-feature-extraction', model='WinKawaks/vit-tiny-patch16-224', model_kwargs={'add_pooling_layer': False})
result = pipe(Image.open("input_files/parrots.png").convert("RGB").resize((320, 320)))
features = np.asarray(result)
print("Feature shape:", features.shape)
print("First values:", features.reshape(-1)[:8])

# Release this example model before opening the next example.
del pipe
gc.collect()

## 08. Text Classification

**Lightweight example.**

**Category:** Natural Language Processing  
**What it does:** Assigns a learned label to a text.  
**API:** Transformers pipeline  
**Runtime:** CPU supported; no GPU required.



[Official reference](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines)

Smaller checkpoints prioritise download size and speed over accuracy. [Model references](#smaller-model-references).

### Example 1: The service was excellent and I am very happy.

Output: POSITIVE or NEGATIVE sentiment and a model confidence score.

In [ ]:
pipe = pipeline('text-classification', model='distilbert/distilbert-base-uncased-finetuned-sst-2-english')
result = pipe('The service was excellent and I am very happy.')
print(result)

# Release this example model before opening the next example.
del pipe
gc.collect()

### Example 2: My card is blocked and support has been unhelpful.

Output: POSITIVE or NEGATIVE sentiment and a model confidence score.

In [ ]:
pipe = pipeline('text-classification', model='distilbert/distilbert-base-uncased-finetuned-sst-2-english')
result = pipe('My card is blocked and support has been unhelpful.')
print(result)

# Release this example model before opening the next example.
del pipe
gc.collect()

## 09. Token Classification

**Lightweight example.**

**Category:** Natural Language Processing  
**What it does:** Assigns labels to words or word spans, such as people and places.  
**API:** Transformers pipeline  
**Runtime:** CPU supported; no GPU required.



[Official reference](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines)

Smaller checkpoints prioritise download size and speed over accuracy. [Model references](#smaller-model-references).

### Example 1: Ravi works at Microsoft in Hyderabad.

Output: named entities with their text spans, labels and scores.

In [ ]:
pipe = pipeline('token-classification', model='dslim/distilbert-NER', aggregation_strategy='simple')
result = pipe('Ravi works at Microsoft in Hyderabad.')
print(result)

# Release this example model before opening the next example.
del pipe
gc.collect()

### Example 2: Priya visited London to meet the team at Google.

Output: named entities with their text spans, labels and scores.

In [ ]:
pipe = pipeline('token-classification', model='dslim/distilbert-NER', aggregation_strategy='simple')
result = pipe('Priya visited London to meet the team at Google.')
print(result)

# Release this example model before opening the next example.
del pipe
gc.collect()

## 10. Table Question Answering

Uses TAPAS-Small, a model trained to answer questions from table cells. Input cells are converted to strings. Output includes selected cells, coordinates and an answer. This is a pretrained table model, rather than asking a small general-purpose LLM to guess from CSV text. Verify predictions against the table.

**API:** `table-question-answering` pipeline. **Runtime:** CPU.

### Example 1: Which branch has the highest deposits?

Output: a generated text answer; verify it against the displayed table.

In [ ]:
table = pd.read_csv("input_files/branch_table.csv").astype(str)
pipe = pipeline("table-question-answering", model="google/tapas-small-finetuned-wtq")
print(pipe(table=table, query='Which branch has the highest deposits?'))
del pipe
gc.collect()

### Example 2: Which branch has deposits of 90?

Output: a generated text answer; verify it against the displayed table.

In [ ]:
table = pd.read_csv("input_files/branch_table.csv").astype(str)
pipe = pipeline("table-question-answering", model="google/tapas-small-finetuned-wtq")
print(pipe(table=table, query='Which branch has deposits of 90?'))
del pipe
gc.collect()

## 11. Question Answering

**Lightweight example.**

**Category:** Natural Language Processing  
**What it does:** Extracts an answer span from a supplied text context.  
**API:** Transformers pipeline  
**Runtime:** CPU supported; no GPU required.



[Official reference](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines)

Smaller checkpoints prioritise download size and speed over accuracy. [Model references](#smaller-model-references).

### Example 1: What is the minimum balance?

Output: extracted answer plus start/end character offsets. The answer must come from the provided context.

In [ ]:
context = open("input_files/bank_policy.txt").read()
pipe = pipeline('question-answering', model='deepset/minilm-uncased-squad2')
result = pipe(question='What is the minimum balance?', context=context)
print(result)

# Release this example model before opening the next example.
del pipe
gc.collect()

### Example 2: What is the maximum loan amount?

Output: extracted answer plus start/end character offsets. The answer must come from the provided context.

In [ ]:
context = open("input_files/loan_policy.txt").read()
pipe = pipeline('question-answering', model='deepset/minilm-uncased-squad2')
result = pipe(question='What is the maximum loan amount?', context=context)
print(result)

# Release this example model before opening the next example.
del pipe
gc.collect()

## 12. Zero-Shot Classification

**Lightweight example.**

**Category:** Natural Language Processing  
**What it does:** Classifies text using candidate labels you supply.  
**API:** Transformers pipeline  
**Runtime:** CPU supported; no GPU required.



[Official reference](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines)

Smaller checkpoints prioritise download size and speed over accuracy. [Model references](#smaller-model-references).

### Example 1: My debit card was declined at the ATM.

Output: ranked candidate labels and scores; no custom fine-tuning is needed for these labels.

In [ ]:
pipe = pipeline('zero-shot-classification', model='typeform/mobilebert-uncased-mnli')
result = pipe('My debit card was declined at the ATM.', candidate_labels=["card issue", "loan enquiry", "account opening"])
print(result)

# Release this example model before opening the next example.
del pipe
gc.collect()

### Example 2: I want to know whether I qualify for a personal loan.

Output: ranked candidate labels and scores; no custom fine-tuning is needed for these labels.

In [ ]:
pipe = pipeline('zero-shot-classification', model='typeform/mobilebert-uncased-mnli')
result = pipe('I want to know whether I qualify for a personal loan.', candidate_labels=["card issue", "loan enquiry", "account opening"])
print(result)

# Release this example model before opening the next example.
del pipe
gc.collect()

## 13. Translation

**Lightweight example.**

**Category:** Natural Language Processing  
**What it does:** Converts text from one language to another.  
**API:** Transformers pipeline  
**Runtime:** CPU supported; no GPU required.



[Official reference](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines)

Smaller checkpoints prioritise download size and speed over accuracy. [Model references](#smaller-model-references).

Fast version: FLAN-T5-Small receives a translation instruction and is reused by other text tasks.

In [ ]:
# Translation reuses FLAN-T5-Small; no extra installation is needed.

### Example 1: Please open a savings account.

Input: English. Output: French text. FLAN-T5-Small follows a translation instruction; verify the translation.

In [ ]:
pipe = pipeline("text2text-generation", model="google/flan-t5-small")
print(pipe("Translate English to French: " + 'Please open a savings account.', max_new_tokens=30, do_sample=False)[0]["generated_text"])

# Release this example model before opening the next example.
del pipe
gc.collect()


### Example 2: The loan payment is due tomorrow.

Input: English. Output: French text. FLAN-T5-Small follows a translation instruction; verify the translation.

In [ ]:
pipe = pipeline("text2text-generation", model="google/flan-t5-small")
print(pipe("Translate English to French: " + 'The loan payment is due tomorrow.', max_new_tokens=30, do_sample=False)[0]["generated_text"])

# Release this example model before opening the next example.
del pipe
gc.collect()


## 14. Summarization

**Lightweight example.**

**Category:** Natural Language Processing  
**What it does:** Produces a shorter version of a longer text.  
**API:** Transformers pipeline  
**Runtime:** CPU supported; no GPU required.



[Official reference](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines)

Smaller checkpoints prioritise download size and speed over accuracy. [Model references](#smaller-model-references).

Fast version: FLAN-T5-Small with a short summary instruction; no large BART checkpoint.

### Example 1: bank_policy

Input: fictional policy text. Output: a short summary. Verify numerical facts because generative summaries can change them.

In [ ]:
text = open("input_files/bank_policy.txt").read()
pipe = pipeline("text2text-generation", model="google/flan-t5-small")
print(pipe("Summarize briefly: " + text, max_new_tokens=40, do_sample=False)[0]["generated_text"])

# Release this example model before opening the next example.
del pipe
gc.collect()


### Example 2: loan_policy

Input: fictional policy text. Output: a short summary. Verify numerical facts because generative summaries can change them.

In [ ]:
text = open("input_files/loan_policy.txt").read()
pipe = pipeline("text2text-generation", model="google/flan-t5-small")
print(pipe("Summarize briefly: " + text, max_new_tokens=40, do_sample=False)[0]["generated_text"])

# Release this example model before opening the next example.
del pipe
gc.collect()


## 15. Feature Extraction

**Lightweight example.**

**Category:** Natural Language Processing  
**What it does:** Converts text tokens into numerical feature vectors.  
**API:** Transformers pipeline  
**Runtime:** CPU supported; no GPU required.



[Official reference](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines)

Smaller checkpoints prioritise download size and speed over accuracy. [Model references](#smaller-model-references).

### Example 1: I need a savings account.

Output: token-level hidden-state features, not sentiment labels. Do not confuse these with a sentence embedding automatically.

In [ ]:
pipe = pipeline('feature-extraction', model='prajjwal1/bert-tiny')
result = pipe('I need a savings account.')
vectors = np.asarray(result)
print("Token feature shape:", vectors.shape)
print("First feature values:", vectors.reshape(-1)[:8])

# Release this example model before opening the next example.
del pipe
gc.collect()

### Example 2: My loan was approved.

Output: token-level hidden-state features, not sentiment labels. Do not confuse these with a sentence embedding automatically.

In [ ]:
pipe = pipeline('feature-extraction', model='prajjwal1/bert-tiny')
result = pipe('My loan was approved.')
vectors = np.asarray(result)
print("Token feature shape:", vectors.shape)
print("First feature values:", vectors.reshape(-1)[:8])

# Release this example model before opening the next example.
del pipe
gc.collect()

## 16. Text Generation

**Lightweight example.**

Use TinyStories-1M to continue two short story prompts. This very small model demonstrates text continuation; it is not an instruction-following banking assistant.

**API:** `text-generation` pipeline. **Runtime:** CPU.


### Example 1: Short story continuation

Input: Once upon a time, a little girl found a red bird. Output: up to 30 new tokens from a very small story model.

In [ ]:
pipe = pipeline("text-generation", model="roneneldan/TinyStories-1M", tokenizer="EleutherAI/gpt-neo-125M")
print(pipe('Once upon a time, a little girl found a red bird.', max_new_tokens=30, do_sample=False)[0]["generated_text"])

# Release this example model before opening the next example.
del pipe
gc.collect()


### Example 2: Short story continuation

Input: A small cat went to the garden and Output: up to 30 new tokens from a very small story model.

In [ ]:
pipe = pipeline("text-generation", model="roneneldan/TinyStories-1M", tokenizer="EleutherAI/gpt-neo-125M")
print(pipe('A small cat went to the garden and', max_new_tokens=30, do_sample=False)[0]["generated_text"])

# Release this example model before opening the next example.
del pipe
gc.collect()


## 17. Fill-Mask

**Lightweight example.**

**Category:** Natural Language Processing  
**What it does:** Predicts a missing token inside a sentence.  
**API:** Transformers pipeline  
**Runtime:** CPU supported; no GPU required.



[Official reference](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines)

Smaller checkpoints prioritise download size and speed over accuracy. [Model references](#smaller-model-references).

### Example 1: The customer deposited money in the [MASK].

Input: text containing [MASK]. Output: likely replacement tokens with scores.

In [ ]:
pipe = pipeline('fill-mask', model='prajjwal1/bert-tiny')
result = pipe('The customer deposited money in the [MASK].', top_k=3)
print(result)

# Release this example model before opening the next example.
del pipe
gc.collect()

### Example 2: The loan payment is due on [MASK].

Input: text containing [MASK]. Output: likely replacement tokens with scores.

In [ ]:
pipe = pipeline('fill-mask', model='prajjwal1/bert-tiny')
result = pipe('The loan payment is due on [MASK].', top_k=3)
print(result)

# Release this example model before opening the next example.
del pipe
gc.collect()

## 18. Sentence Similarity

**Lightweight example.**

**Category:** Natural Language Processing  
**What it does:** Compares the meaning of two sentences.  
**API:** Sentence Transformers direct API  
**Runtime:** CPU supported; no GPU required.



[Official reference](https://huggingface.co/sentence-transformers/paraphrase-MiniLM-L3-v2)

Smaller checkpoints prioritise download size and speed over accuracy. [Model references](#smaller-model-references).

### Example 1: How can I open a bank account? / What is the process for opening an account?

Output: cosine similarity between sentence embeddings. A higher score indicates more similar meaning; it is not an accuracy percentage.

In [ ]:
from sentence_transformers import SentenceTransformer
pipe = SentenceTransformer("sentence-transformers/paraphrase-MiniLM-L3-v2")
vectors = pipe.encode(['How can I open a bank account?', 'What is the process for opening an account?'], normalize_embeddings=True)
score = float(vectors[0] @ vectors[1])
print("Cosine similarity:", round(score, 3))

# Release this example model before opening the next example.
del pipe
gc.collect()

### Example 2: My card is blocked. / The weather is sunny today.

Output: cosine similarity between sentence embeddings. A higher score indicates more similar meaning; it is not an accuracy percentage.

In [ ]:
from sentence_transformers import SentenceTransformer
pipe = SentenceTransformer("sentence-transformers/paraphrase-MiniLM-L3-v2")
vectors = pipe.encode(['My card is blocked.', 'The weather is sunny today.'], normalize_embeddings=True)
score = float(vectors[0] @ vectors[1])
print("Cosine similarity:", round(score, 3))

# Release this example model before opening the next example.
del pipe
gc.collect()

## 19. Text Ranking

**Lightweight example.**

**Category:** Natural Language Processing  
**What it does:** Ranks candidate passages for a search query.  
**API:** Sentence Transformers CrossEncoder  
**Runtime:** CPU supported; no GPU required.



[Official reference](https://huggingface.co/cross-encoder/ms-marco-TinyBERT-L2-v2)

Smaller checkpoints prioritise download size and speed over accuracy. [Model references](#smaller-model-references).

### Example 1: How do I replace my debit card?

Output: passages ranked by query relevance. Cross-encoder scores are relevance logits, not calibrated probabilities.

In [ ]:
from sentence_transformers import CrossEncoder
pipe = CrossEncoder("cross-encoder/ms-marco-TinyBERT-L2-v2")
query = 'How do I replace my debit card?'
passages = [
    "Debit card replacement costs INR 200. Contact the bank branch.",
    "Personal loan annual interest is 12 percent.",
    "The bank branch opens at 9 AM."
]
scores = pipe.predict([[query, passage] for passage in passages])
print(pd.DataFrame({"passage": passages, "score": scores}).sort_values("score", ascending=False))

# Release this example model before opening the next example.
del pipe
gc.collect()

### Example 2: What is the personal loan interest rate?

Output: passages ranked by query relevance. Cross-encoder scores are relevance logits, not calibrated probabilities.

In [ ]:
from sentence_transformers import CrossEncoder
pipe = CrossEncoder("cross-encoder/ms-marco-TinyBERT-L2-v2")
query = 'What is the personal loan interest rate?'
passages = [
    "Debit card replacement costs INR 200. Contact the bank branch.",
    "Personal loan annual interest is 12 percent.",
    "The bank branch opens at 9 AM."
]
scores = pipe.predict([[query, passage] for passage in passages])
print(pd.DataFrame({"passage": passages, "score": scores}).sort_values("score", ascending=False))

# Release this example model before opening the next example.
del pipe
gc.collect()

## 20. Text-to-Speech

**Lightweight example.**

**Category:** Audio  
**What it does:** Converts written text into spoken audio.  
**API:** Transformers pipeline  
**Runtime:** CPU supported; no GPU required.



[Official reference](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines)

Smaller checkpoints prioritise download size and speed over accuracy. [Model references](#smaller-model-references).

### Example 1: Welcome to Demo Bank. How can I help you?

Input: text. Output: synthesised speech waveform and sample rate.

In [ ]:
pipe = pipeline('text-to-speech', model='facebook/mms-tts-eng')
result = pipe('Welcome to Demo Bank. How can I help you?')
display(Audio(np.asarray(result["audio"]).squeeze(), rate=result["sampling_rate"]))
sf.write("outputs/speech_output_1.wav", np.asarray(result["audio"]).reshape(-1), result["sampling_rate"])

# Release this example model before opening the next example.
del pipe
gc.collect()

### Example 2: Your card replacement request has been received.

Input: text. Output: synthesised speech waveform and sample rate.

In [ ]:
pipe = pipeline('text-to-speech', model='facebook/mms-tts-eng')
result = pipe('Your card replacement request has been received.')
display(Audio(np.asarray(result["audio"]).squeeze(), rate=result["sampling_rate"]))
sf.write("outputs/speech_output_2.wav", np.asarray(result["audio"]).reshape(-1), result["sampling_rate"])

# Release this example model before opening the next example.
del pipe
gc.collect()

## 22. Audio-to-Audio

**Lightweight example.**

**Category:** Audio  
**What it does:** Transforms an audio waveform into another waveform.  
**API:** Direct Encodec API  
**Runtime:** CPU supported; no GPU required.



[Official reference](https://huggingface.co/docs/transformers/v4.57.1/en/model_doc/encodec)

Smaller checkpoints prioritise download size and speed over accuracy. [Model references](#smaller-model-references).

### Example 1: Reconstruct speech segment 1

Input: speech. Output: codec-reconstructed speech. This is neural audio compression/reconstruction, not denoising or voice conversion.

In [ ]:
from transformers import AutoProcessor, EncodecModel
processor = AutoProcessor.from_pretrained("facebook/encodec_24khz")
pipe = EncodecModel.from_pretrained("facebook/encodec_24khz").eval()
audio, rate = librosa.load("input_files/speech_1.wav", sr=24000)
inputs = processor(raw_audio=audio, sampling_rate=rate, return_tensors="pt")
with torch.inference_mode():
    codes = pipe.encode(inputs["input_values"], inputs["padding_mask"])
    reconstructed = pipe.decode(codes.audio_codes, codes.audio_scales, inputs["padding_mask"])
waveform = reconstructed.audio_values[0, 0].cpu().numpy()
sf.write("outputs/reconstructed_1.wav", waveform, rate)
display(Audio(waveform, rate=rate))
print("Compressed-code shape:", tuple(codes.audio_codes.shape))
del inputs, codes, reconstructed

# Release this example model before opening the next example.
del pipe
gc.collect()

### Example 2: Reconstruct speech segment 2

Input: speech. Output: codec-reconstructed speech. This is neural audio compression/reconstruction, not denoising or voice conversion.

In [ ]:
from transformers import AutoProcessor, EncodecModel
processor = AutoProcessor.from_pretrained("facebook/encodec_24khz")
pipe = EncodecModel.from_pretrained("facebook/encodec_24khz").eval()
audio, rate = librosa.load("input_files/speech_2.wav", sr=24000)
inputs = processor(raw_audio=audio, sampling_rate=rate, return_tensors="pt")
with torch.inference_mode():
    codes = pipe.encode(inputs["input_values"], inputs["padding_mask"])
    reconstructed = pipe.decode(codes.audio_codes, codes.audio_scales, inputs["padding_mask"])
waveform = reconstructed.audio_values[0, 0].cpu().numpy()
sf.write("outputs/reconstructed_2.wav", waveform, rate)
display(Audio(waveform, rate=rate))
print("Compressed-code shape:", tuple(codes.audio_codes.shape))
del inputs, codes, reconstructed

# Release this example model before opening the next example.
del pipe
gc.collect()

## 23. Voice Activity Detection

**Lightweight example.**

**Category:** Audio  
**What it does:** Finds intervals containing speech.  
**API:** Silero VAD with Hugging Face ONNX weights  
**Runtime:** CPU supported; no GPU required.



[Official reference](https://huggingface.co/onnx-community/silero-vad)

Smaller checkpoints prioritise download size and speed over accuracy. [Model references](#smaller-model-references).

### Example 1: speech_1.wav

Input: audio. Output: start/end times for detected speech. Silence should usually give an empty list. This does not transcribe words.

In [ ]:
from huggingface_hub import hf_hub_download
from silero_vad import get_speech_timestamps
from silero_vad.utils_vad import OnnxWrapper
model_file = hf_hub_download("onnx-community/silero-vad", filename="onnx/model.onnx")
pipe = OnnxWrapper(model_file, force_onnx_cpu=True)
audio, _ = librosa.load("input_files/speech_1.wav", sr=16000)
segments = get_speech_timestamps(torch.tensor(audio), pipe, sampling_rate=16000, return_seconds=True)
print("Speech regions in seconds:", segments)

# Release this example model before opening the next example.
del pipe
gc.collect()

### Example 2: silence.wav

Input: audio. Output: start/end times for detected speech. Silence should usually give an empty list. This does not transcribe words.

In [ ]:
from huggingface_hub import hf_hub_download
from silero_vad import get_speech_timestamps
from silero_vad.utils_vad import OnnxWrapper
model_file = hf_hub_download("onnx-community/silero-vad", filename="onnx/model.onnx")
pipe = OnnxWrapper(model_file, force_onnx_cpu=True)
audio, _ = librosa.load("input_files/silence.wav", sr=16000)
segments = get_speech_timestamps(torch.tensor(audio), pipe, sampling_rate=16000, return_seconds=True)
print("Speech regions in seconds:", segments)

# Release this example model before opening the next example.
del pipe
gc.collect()

## 24. Time Series Forecasting

**Lightweight example.**

**Category:** Tabular  
**What it does:** Predicts future values from an ordered history.  
**API:** ChronosPipeline from the chronos-forecasting library  
**Runtime:** CPU supported; no GPU required.



[Official reference](https://huggingface.co/amazon/chronos-t5-tiny)

Smaller checkpoints prioritise download size and speed over accuracy. [Model references](#smaller-model-references).

### Example 1: Transactions

Input: the earlier daily observations only. Output: forecast distribution and median forecast. The last 14 days stay unseen until evaluation.

In [ ]:
from chronos import ChronosPipeline
from sklearn.metrics import mean_absolute_error
pipe = ChronosPipeline.from_pretrained("amazon/chronos-t5-tiny", device_map="cpu", torch_dtype=torch.float32)
series = pd.read_csv("input_files/daily_series.csv")['transactions'].to_numpy(dtype="float32")
context = torch.tensor(series[:-14])
forecast = pipe.predict(context, prediction_length=14, num_samples=5)
median = torch.quantile(forecast[0].float(), 0.5, dim=0).numpy()
print("Future 14-day median forecast:", median.round(1))
print("Held-out MAE:", mean_absolute_error(series[-14:], median))
print(pd.DataFrame({"actual": series[-14:], "forecast": median}))

# Release this example model before opening the next example.
del pipe
gc.collect()

### Example 2: Atm Withdrawals

Input: the earlier daily observations only. Output: forecast distribution and median forecast. The last 14 days stay unseen until evaluation.

In [ ]:
from chronos import ChronosPipeline
from sklearn.metrics import mean_absolute_error
pipe = ChronosPipeline.from_pretrained("amazon/chronos-t5-tiny", device_map="cpu", torch_dtype=torch.float32)
series = pd.read_csv("input_files/daily_series.csv")['atm_withdrawals'].to_numpy(dtype="float32")
context = torch.tensor(series[:-14])
forecast = pipe.predict(context, prediction_length=14, num_samples=5)
median = torch.quantile(forecast[0].float(), 0.5, dim=0).numpy()
print("Future 14-day median forecast:", median.round(1))
print("Held-out MAE:", mean_absolute_error(series[-14:], median))
print(pd.DataFrame({"actual": series[-14:], "forecast": median}))

# Release this example model before opening the next example.
del pipe
gc.collect()

## Smaller model references

- [distilbert/distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english)
- [WinKawaks/vit-tiny-patch16-224](https://huggingface.co/WinKawaks/vit-tiny-patch16-224)
- [cross-encoder/ms-marco-TinyBERT-L2-v2](https://huggingface.co/cross-encoder/ms-marco-TinyBERT-L2-v2)
- [dslim/distilbert-NER](https://huggingface.co/dslim/distilbert-NER)
- [google/flan-t5-small](https://huggingface.co/google/flan-t5-small)
- [deepset/minilm-uncased-squad2](https://huggingface.co/deepset/minilm-uncased-squad2)
- [nvidia/segformer-b0-finetuned-ade-512-512](https://huggingface.co/nvidia/segformer-b0-finetuned-ade-512-512)
- [prajjwal1/bert-tiny](https://huggingface.co/prajjwal1/bert-tiny)
- [roneneldan/TinyStories-1M](https://huggingface.co/roneneldan/TinyStories-1M)
- [sentence-transformers/paraphrase-MiniLM-L3-v2](https://huggingface.co/sentence-transformers/paraphrase-MiniLM-L3-v2)
- [typeform/mobilebert-uncased-mnli](https://huggingface.co/typeform/mobilebert-uncased-mnli)